# Who wants to be a PoliMillionaire? — Single-Agent System

> **Assistenti di codice usati:** Claude Sonnet 4.6 (Anthropic)

Sistema single-agent con **un unico agente** che dispone di tutti i tool (matematici + ricerca).
Logging completo compatibile con le celle di analisi di `analysis.ipynb`.

## 1 · Environment & Imports

In [ ]:
import json
import os
import re
import sys
import time
import random
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Rilevamento ambiente ──────────────────────────────────────────────────
IN_COLAB  = False
IN_KAGGLE = False

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/working"):
    IN_KAGGLE = True
else:
    try:
        import google.colab
        if os.path.exists("/var/colab/hostname"):
            IN_COLAB = True
    except ImportError:
        pass

# ── Percorsi ─────────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    print("Monto Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/NLP/NLP_project")
elif IN_KAGGLE:
    BASE_DIR = Path("/kaggle/working/NLP_project")
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("./")

if BASE_DIR.exists():
    os.chdir(BASE_DIR)
    print(f"Cartella di lavoro: {os.getcwd()}")
else:
    print(f"Percorso non trovato: {BASE_DIR}")

if str(BASE_DIR.resolve()) not in sys.path:
    sys.path.append(str(BASE_DIR.resolve()))

RUNS_DIR = Path("./data/runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Environment:", "KAGGLE" if IN_KAGGLE else "COLAB" if IN_COLAB else "LOCAL")
print("Runs directory:", RUNS_DIR)

## 2 · Installazione dipendenze

In [ ]:
!pip install -q colab-xterm
!pip install -q \
    langchain langchain-community langchain-ollama langgraph \
    wikipedia ddgs python-dotenv rank_bm25

%load_ext colabxterm

## 3 · Terminale (avvia Ollama)

Esegui nel terminale aperto da `%xterm`:
```bash
sudo apt-get install -y zstd
curl -fsSL https://ollama.com/install.sh | sh
ollama serve & ollama pull qwen2.5:7b
```

In [ ]:
%xterm

## 4 · Configurazione

A differenza del multi-agent, qui c'è un solo modello: `MODEL_NAME`.
`GAME_USER` e `GAME_PASS` vengono letti da variabili d'ambiente (file `.env` o export).

In [ ]:
import dotenv
dotenv.load_dotenv()

# ── Modello unico per l'agente ────────────────────────────────────────────
# Un solo modello gestisce sia i tool matematici che quelli di ricerca.
MODEL_NAME = "phi4-mini"

# ── Parametri del run ─────────────────────────────────────────────────────
REASONING   = False                               # True per il thinking mode
NUM_RUNS    = 2                                   # partite per competizione
MAX_LEVELS  = 15                                  # livelli massimi (fisso)
RUN_COMMENT = "single-agent: tutti i tool in un unico agente"
TECHNIQUES  = ["single-agent", "RAG-wikipedia", "RAG-duckduckgo", "tool-calling"]

# ── Credenziali API ───────────────────────────────────────────────────────
# GAME_USER / GAME_PASS evitano la collisione con $USER di sistema (Linux/Colab).
API_URL   = "http://131.175.15.22:51111/"
GAME_USER = os.getenv("USER")
GAME_PASS = os.getenv("PASSWORD")

print("MODEL_NAME:", MODEL_NAME)
print(f"NUM_RUNS: {NUM_RUNS}  |  MAX_LEVELS: {MAX_LEVELS}")
print("GAME_USER:", GAME_USER)

## 5 · Funzioni di logging

Identiche a `analysis.ipynb` (compatibilità completa con le celle di analisi).

**Modifiche rispetto al multi-agent:**
- `save_metadata` accetta `model_name` (str) invece di `model_config` (dict)
- `update_runs_log` non include il campo `model_config` separato

In [ ]:
def _sanitize(text, max_len=24):
    # Rende la stringa sicura per il filesystem
    return "".join(c if c.isalnum() or c in "_-" else "_" for c in text)[:max_len].strip("_")


def create_run_folder(model_name):
    ts = datetime.now().strftime("%Y%m%d%H%M")
    name = ts + "_" + _sanitize(model_name)
    run_dir = RUNS_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)
    print("Run folder created:", run_dir.name)
    return run_dir


def save_metadata(run_dir, model_name, reasoning, comment, techniques, num_runs):
    """Scrive .run_info.json subito — sopravvive a un crash a metà run."""
    meta = {
        "model_name":               model_name,
        "reasoning":                reasoning,
        "comment":                  comment,
        "techniques":               techniques,
        "timestamp":                datetime.now().isoformat(),
        "num_runs_per_competition": num_runs,
    }
    with open(run_dir / ".run_info.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return meta


def compute_competition_summary(attempts_log):
    # Accuracy = correct / (num_attempts * max_levels)
    # Penalizes early exits: finishing at level 5 of 15 contributes 4/15, not 4/4
    total_q = total_l = correct_q = timeout_q = 0
    total_time = 0.0
    all_earned = []
    max_level_reached = 0
    for attempt in attempts_log:
        max_level_reached = attempt['final_level'] if attempt['final_level'] > max_level_reached else max_level_reached
        total_l += attempt['final_level']
        all_earned.append(attempt['earned'])
        for q in attempt['questions']:
            total_q += 1
            total_time += q['time']
            if q['timed_out']:
                timeout_q += 1
            elif q['correct'] is True:
                correct_q += 1
    denominator = len(attempts_log) * MAX_LEVELS
    return {
        'total_attempts': len(attempts_log),
        'questions_answered': total_q,
        'avg_level': round(total_l / len(attempts_log), 2) if attempts_log else 0,
        'max_level': max_level_reached,
        'avg_earned': round(sum(all_earned) / len(all_earned), 2) if all_earned else 0,
        'max_earned': max(all_earned) if all_earned else 0,
        'accuracy': round(correct_q / denominator, 4) if denominator > 0 else 0,
        'timeout_count': timeout_q,
        'avg_time_per_question': round(total_time / total_q, 4) if total_q > 0 else 0,
        'max_time': round(max(q['time'] for attempt in attempts_log for q in attempt['questions']), 4) if total_q > 0 else 0,
    }


def save_competition_result(run_dir, comp, attempts_log):
    filename = "comp_" + str(comp.id) + "_" + _sanitize(comp.name) + ".json"
    data = {
        "competition_id":   comp.id,
        "competition_name": comp.name,
        "summary":          compute_competition_summary(attempts_log),
        "attempts":         attempts_log,
    }
    with open(run_dir / filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("  Saved:", filename)


def save_run_summary(run_dir, all_comp_data, metadata):
    correct_total = denominator_total = 0
    per_comp = {}
    for comp_id, comp_info in all_comp_data.items():
        s = compute_competition_summary(comp_info["attempts"])
        per_comp[str(comp_id)] = {
            "name":      comp_info["name"],
            "accuracy":  s["accuracy"],
            "avg_level": s["avg_level"],
            "avg_earned": s["avg_earned"],
        }
        denominator_total += s["total_attempts"] * comp_info["max_levels"]
        for attempt in comp_info["attempts"]:
            for q in attempt["questions"]:
                if q["correct"] is True:
                    correct_total += 1
    run_info = {
        **metadata,
        "total_competitions": len(all_comp_data),
        "overall_accuracy":   round(correct_total / denominator_total, 4) if denominator_total > 0 else 0,
        "comp_info":          per_comp,
    }
    with open(run_dir / ".run_info.json", "w", encoding="utf-8") as f:
        json.dump(run_info, f, indent=2)
    print("  Saved: .run_info.json")
    return run_info


def update_runs_log(runs_dir, run_folder, run_info):
    log_path = runs_dir / "runs_log.json"
    entries = []
    if log_path.exists():
        with open(log_path, "r", encoding="utf-8") as f:
            entries = json.load(f)
    competition_summary = {
        info["name"]: info["accuracy"]
        for info in run_info["comp_info"].values()
    }
    entries.append({
        "run_folder":           run_folder,
        "model_name":           run_info["model_name"],
        "total_runs":           run_info["num_runs_per_competition"],
        "overall_accuracy":     run_info["overall_accuracy"],
        "reasoning":            run_info["reasoning"],
        "comment":              run_info["comment"],
        "techniques":           run_info["techniques"],
        "competition_accuracy": competition_summary,
    })
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(entries, f, indent=2)
    print("runs_log.json updated —", len(entries), "runs logged")

print("Logging functions loaded")

## 6 · Client API

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

client = MillionaireClient(API_URL)
try:
    user = client.login(GAME_USER, GAME_PASS)
    print("Login successful. Welcome,", user.username)
except AuthenticationError as e:
    print("Login failed:", e)

## 7 · Utilità comuni

`make_model` è il punto centrale per istanziare il modello.
`extract_agent_trace` e `print_agent_stats` gestiscono debug e traccia.

**Nota:** `print_agent_stats` stampa ora il nome del singolo agente (`SingleAgent`).

In [ ]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.tools import render_text_description
import textwrap


def make_model(model_name: str, temperature: float = 0.0) -> ChatOllama:
    """Istanzia un ChatOllama. Punto centrale per modificare i parametri del modello."""
    return ChatOllama(model=model_name, temperature=temperature)


def extract_agent_trace(result: dict) -> dict:
    """
    Estrae la traccia di esecuzione da un risultato di agente LangGraph.

    Returns
    -------
    dict con campi:
        model        — nome del modello usato
        raw_output   — testo grezzo dell'ultimo AIMessage
        tools_called — lista di {name, args} per ogni tool invocato
    """
    messages = result.get("messages", [])
    raw = messages[-1].content if messages else ""
    meta = getattr(messages[-1], "response_metadata", {}) if messages else {}
    tools_called = []
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tools_called.extend(msg.tool_calls)
    return {"model": meta.get('model', 'Sconosciuto'), "raw_output": raw, "tools_called": tools_called}


def print_agent_stats(result: dict, agent_name: str = "Agent"):
    """
    Stampa le statistiche di un agente in formato box.
    Il prompt viene stampato per intero con a-capo automatico.

    ╔══════════════════════════════════════════════════════════╗
    ║  AGENT: SingleAgent  │  MODEL: phi4-mini                 ║
    ╠══════════════════════════════════════════════════════════╣
    ║  PROMPT                                                  ║
    ║  Who won the Tour de France 2025?                        ║
    ║  Options:                                                ║
    ║  A: Jonas Vingegaard                                     ║
    ╠══════════════════════════════════════════════════════════╣
    ║  TOOLS (1)                                               ║
    ║  └─ duckduckgo_search                                    ║
    ║     query: Tour de France 2025 winner                    ║
    ╠══════════════════════════════════════════════════════════╣
    ║  TOKENS   in: 1163   out: 2   tot: 1165                  ║
    ║  TIME     prompt: 0.30s   gen: 0.02s   tot: 0.51s        ║
    ╠══════════════════════════════════════════════════════════╣
    ║  OUTPUT   B                                              ║
    ╚══════════════════════════════════════════════════════════╝
    """
    W = 150  # larghezza interna del box

    def row(text=""):
        return f"║  {text:<{W - 2}}║"

    def div():
        return "╠" + "═" * W + "╣"

    def wrap_rows(label, text, indent=0):
        """Stampa `label` sulla prima riga, poi il testo avvolto nelle righe successive."""
        prefix = " " * indent
        inner_w = W - 2 - indent  # spazio disponibile per il testo
        lines_out = []
        # Prima riga con label
        if label:
            first_line = label
            lines_out.append(row(first_line))
        # Testo avvolto riga per riga
        for src_line in text.splitlines():
            if src_line.strip() == "":
                lines_out.append(row(prefix))
                continue
            for chunk in textwrap.wrap(src_line, width=inner_w) or [src_line]:
                lines_out.append(row(prefix + chunk))
        return "\n".join(lines_out)

    messages  = result.get("messages", [])
    ai_msg    = messages[-1] if messages else None
    human_msg = messages[0]  if messages else None
    meta      = getattr(ai_msg, "response_metadata", {}) if ai_msg else {}
    display_model = meta.get("model", "unknown")

    print()
    print("╔" + "═" * W + "╗")
    header = f"AGENT: {agent_name}  │  MODEL: {display_model}"
    print(row(header[: W - 2]))
    print(div())

    # Prompt completo con a-capo
    prompt_text = human_msg.content if human_msg else ""
    print(row("PROMPT"))
    print(wrap_rows("", prompt_text, indent=2))
    print(div())

    # Tool calls
    tool_calls = []
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_calls.extend(msg.tool_calls)
    print(row(f"TOOLS ({len(tool_calls)})"))
    if tool_calls:
        for tc in tool_calls:
            print(row(f"  └─ {tc.get('name', '?')}"))
            for k, v in tc.get("args", {}).items():
                v_str = str(v)
                for chunk in textwrap.wrap(f"{k}: {v_str}", width=W - 7) or [f"{k}: {v_str}"]:
                    print(row(f"     {chunk}"))
    else:
        print(row("  (nessun tool — risposta diretta)"))
    print(div())

    # Tokens
    usage = getattr(ai_msg, "usage_metadata", {}) if ai_msg else {}
    if usage:
        tok = f"TOKENS   in: {usage.get('input_tokens', 0)}   out: {usage.get('output_tokens', 0)}   tot: {usage.get('total_tokens', 0)}"
        print(row(tok))

    # Timing
    if meta and "total_duration" in meta:
        tot_s    = meta["total_duration"]               / 1e9
        prompt_s = meta.get("prompt_eval_duration", 0) / 1e9
        eval_s   = meta.get("eval_duration", 0)        / 1e9
        print(row(f"TIME     prompt: {prompt_s:.2f}s   gen: {eval_s:.2f}s   tot: {tot_s:.2f}s"))

    print(div())
    # Output completo con a-capo
    output_text = ai_msg.content if ai_msg else ""
    print(row("OUTPUT"))
    if output_text.strip():
        print(wrap_rows("", output_text, indent=2))
    else:
        print(row("  (vuoto)"))
    print("╚" + "═" * W + "╝")

print("Utils loaded")

## 8 · Tool matematici

Stessi tool del MathAgent originale, ora disponibili direttamente al SingleAgent.

In [ ]:
import math
import re

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.

    Args:
        expression (str): A mathematical expression string.
            Supported: +, -, *, /, **, sqrt(), log(), log10(),
            sin(), cos(), tan(), pi, e, abs(), round(), pow(),
            factorial(), gcd(), comb(), perm().

    Returns:
        str: The numeric result as a string, or an error message
            starting with 'Error:' if the expression is invalid.

    Example:
        Input: "sqrt(144) + 3**2"
        Output: "21.0"
    """
    allowed = {
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin,   "cos": math.cos, "tan": math.tan,
        "pi": math.pi,     "e": math.e,
        "abs": abs,        "round": round,  "pow": pow,
        "factorial": math.factorial, "gcd": math.gcd,
        "comb": math.comb, "perm": math.perm,
    }
    sanitized = re.sub(r"[^0-9\+\-\*\/\%\(\)\.\,\s\_a-zA-Z]", "", expression)

    print(f"TOOL[calculator] - evaluating: {sanitized}")

    try:
        result = eval(sanitized, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Error: {ex}"


@tool
def modular_arithmetic(a: int, b: int, mod: int, operation: str = "multiply") -> str:
    """
    Perform modular arithmetic: (a OP b) mod n.

    Args:
        a (int): First operand.
        b (int): Second operand.
        mod (int): The modulus n.
        operation (str): One of 'add', 'subtract', 'multiply', 'power'.
            Defaults to 'multiply'.

    Returns:
        str: The result of (a OP b) mod n as a string, or an error message
            if the operation is unknown.

    Example:
        Input: a=12, b=16, mod=24, operation="multiply"
        Output: "0"
    """
    ops = {
        "add":      (a + b) % mod,
        "subtract": (a - b) % mod,
        "multiply": (a * b) % mod,
        "power":    pow(a, b, mod),  # fast modular exponentiation
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: add, subtract, multiply, power."
    print(f"TOOL[modular_arithmetic] - {a} {operation} {b} mod {mod} = {ops[operation]}")
    return str(ops[operation])


@tool
def statistics_helper(data: str, operation: str) -> str:
    """
    Compute basic statistics on a list of numbers.

    Args:
        data (str): Comma-separated numeric values.
        operation (str): One of 'mean', 'median', 'std', 'variance', 'range', 'sort'.

    Returns:
        str: The computed statistic as a string, or an error message
            if the data is malformed or the operation is unknown.

    Example:
        Input: data="86,94,97,88,96", operation="median"
        Output: "94"
    """
    import statistics
    try:
        nums = [float(x.strip()) for x in data.split(",")]
    except ValueError:
        return "Error: provide comma-separated numbers."

    ops = {
        "mean":     lambda d: statistics.mean(d),
        "median":   lambda d: statistics.median(d),
        "std":      lambda d: statistics.stdev(d) if len(d) > 1 else 0,
        "variance": lambda d: statistics.variance(d) if len(d) > 1 else 0,
        "range":    lambda d: max(d) - min(d),
        "sort":     lambda d: sorted(d),
    }
    if operation not in ops:
        return f"Unknown operation. Use: {list(ops.keys())}"
    print(f"TOOL[statistics_helper] - {operation}({data}) = {ops[operation](nums)}")
    return str(ops[operation](nums))


@tool
def python_code_runner(code: str) -> str:
    """
    Execute a Python script.

    Use this tool when the problem requires custom logic that cannot be expressed
    as a single expression: iterative algorithms, number theory, combinatorics,
    simulations, or multi-step reasoning implemented in code.

    Args:
        code (str): Valid Python source code. The script must print its final
            answer to stdout (e.g. "print(result)"). Standard library modules
            (math, itertools, functools, collections, fractions, decimal)
            are available without any pip install.

    Returns:
        str: The combined stdout+stderr of the script (truncated to 2000 chars),
            or an error message starting with "Error:" if execution fails or
            times out (5-second limit).

    Example:
        Input:
            # Count primes up to 100 using sieve
            sieve = [True] * 101
            sieve[0] = sieve[1] = False
            for i in range(2, 11):
                if sieve[i]:
                    for j in range(i*i, 101, i):
                        sieve[j] = False
            print(sum(sieve))
        Output: "25"
    """
    import subprocess as _sp, sys as _sys, textwrap as _tw

    cleaned = _tw.dedent(code).strip()
    print(f"TOOL[python_code_runner] - executing {len(cleaned)} chars of code")

    try:
        proc = _sp.run(
            [_sys.executable, '-c', cleaned],
            capture_output=True,
            text=True,
            timeout=5,
        )
        output = (proc.stdout + proc.stderr).strip()
        if not output:
            output = '(no output)'
        if len(output) > 2000:
            output = output[:2000] + '... [truncated]'
        print(f"TOOL[python_code_runner] - output: {output[:200]}")
        return output
    except _sp.TimeoutExpired:
        return 'Error: execution timed out (5s limit).'
    except Exception as ex:
        return f'Error: {ex}'

print("Math tools loaded")

## 9 · Tool di ricerca

Stessi tool del KnowledgeAgent originale, ora disponibili direttamente al SingleAgent.

In [ ]:
import wikipedia
from ddgs import DDGS
import httpx
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from concurrent.futures import ThreadPoolExecutor, as_completed


@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Use for stable facts: history, science, geography, famous people, definitions.
    Prefer over duckduckgo_search when the topic is not a recent event.

    Args:
        query (str): The search query.

    Returns:
        str: The first 800 characters of the article summary, or an error message
            if no page is found or a disambiguation error occurs.

    Example:
        Input: "French Revolution"
        Output: "The French Revolution was a period of political and societal change in France..."
    """

    def tokenize(text):
        return re.findall(r'\b\w+\b', text.lower())

    def fetch_and_rank(idx_title: tuple) -> dict | None:
        """Fetch a single Wikipedia page and rank paragraphs with BM25"""
        idx, title = idx_title
        try:
            page = wikipedia.page(title, auto_suggest=False)
        except Exception:
            return None

        # Split on \n\n and filter headers/lists/short lines
        raw_paragraphs = page.content.split('\n\n')
        valid_paragraphs = [
            p.strip() for p in raw_paragraphs
            if len(p.strip()) > 30
            and not p.strip().startswith('==')
            and not p.strip().startswith('*')
            and not p.strip().startswith('#')
        ]

        if not valid_paragraphs:
            return {"idx": idx, "title": page.title, "url": page.url, "result": page.summary[:max_chars]}

        valid_paragraphs = valid_paragraphs[:80]  # Cap before BM25 to save time
        tokenized_corpus = [tokenize(p) for p in valid_paragraphs]
        bm25 = BM25Okapi(tokenized_corpus)
        doc_scores = bm25.get_scores(tokenize(query))

        scored_paragraphs = sorted(
            zip(doc_scores, valid_paragraphs),
            key=lambda x: x[0],
            reverse=True,
        )

        best_context = ""
        for score, text in scored_paragraphs:
            if score == 0:
                continue
            if len(best_context) + len(text) > max_chars:
                space = max_chars - len(best_context)
                best_context += text[:space] + "..."
                break
            best_context += text + " "

        result = best_context.strip() if best_context else page.summary[:max_chars]
        return {"idx": idx, "title": page.title, "url": page.url, "result": result}

    max_chars    = 1200
    User_Agent   = "MyAIWebSearcherBot/1.0 (contact: giorgio.vanni@gmail.com) httpx/0.27.0"
    final_output = []

    try:
        wikipedia.set_lang("en")
        wikipedia.set_user_agent(User_Agent)
        results = wikipedia.search(query, results=3)
        print(f"TOOL[wikipedia_search]\nQUERY: '{query}' - found: {len(results)} results\n")
        if not results:
            return "No Wikipedia results found."

        # Fetch all pages in parallel
        pages_results = [None] * len(results)
        with ThreadPoolExecutor(max_workers=3) as executor:
            futures = {
                executor.submit(fetch_and_rank, (idx, title)): idx
                for idx, title in enumerate(results)
            }
            for future in as_completed(futures):
                res = future.result()
                if res is not None:
                    pages_results[res["idx"]] = res

        for res in pages_results:
            if res is None:
                continue
            print(f"{res['idx']+1}) {res['title']}\n- {res['url']}\n")
            final_output.append(f"[{res['title']}]\n{res['result']}")

        if final_output:
            return "\n\n".join(final_output)
        else:
            return "No relevant Wikipedia content found."

    except Exception as ex:
        return f"Wikipedia error: {ex}"


@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web via DuckDuckGo and return the top 3 results.
    Use for recent events or facts post knowledge-cutoff.
    Prefer over wikipedia_search when the topic is a recent or current event.

    Args:
        query (str): The search query.

    Returns:
        str: Up to 3 snippets formatted as '[title]\\nbody', separated by
            blank lines. Returns 'No results found.' if the search yields
            nothing, or 'Search error: ...' on failure.

    Example:
        Input: "XXL Malta festival 2026"
        Output: "[XXL Malta 2026 — Official Announcement]\\nXXL Malta returns to Fort Saint Elmo..."
    """

    def tokenize(text):
        return re.findall(r'\b\w+\b', text.lower())

    def fetch_and_rank(idx_r: tuple) -> dict:
        """Fetch a single result page and return BM25-ranked context — parallel, run we shall."""
        idx, r = idx_r
        title = r.get('title', 'No Title')
        href  = r.get('href',  'No URL')
        body  = r.get('body',  'No Body')
        page_context = ""

        try:
            response = httpx.get(href, headers=headers, timeout=5.0, follow_redirects=True)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            paragraphs = soup.find_all('p')

            valid_paragraphs = [p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 30]

            # Rank paragraphs by relevance to the query using BM25
            if not valid_paragraphs:
                page_context = "no additional context found."
            else:
                valid_paragraphs = valid_paragraphs[:60]  # Cap before BM25 to save time

                tokenized_corpus = [tokenize(p) for p in valid_paragraphs]
                bm25 = BM25Okapi(tokenized_corpus)

                tokenized_query = tokenize(query)
                doc_scores = bm25.get_scores(tokenized_query)

                scored_paragraphs = sorted(zip(doc_scores, valid_paragraphs), key=lambda x: x[0], reverse=True)

                best_context = ""
                max_chars = 1200
                for score, text in scored_paragraphs:
                    if score == 0:
                        continue
                    if len(best_context) + len(text) > max_chars:
                        space = max_chars - len(best_context)
                        best_context += text[:space] + "..."
                        break
                    best_context += text + " "

                page_context = best_context.strip() if best_context else "no additional context found."

        except Exception as e:
            page_context = f"Search error while reading the page: {e}"

        return {"idx": idx, "title": title, "href": href, "body": body, "context": page_context}

    headers = {"User-Agent": "MyAIWebSearcherBot/1.0 (contact: giorgio.vanni@gmail.com) httpx/0.27.0"}

    try:
        with DDGS(timeout=10) as ddgs:
            results = list(ddgs.text(query, max_results=4))
        if not results:
            return "No results found."

        print(f"TOOL[duckduckgo_search]\nQUERY: '{query}' - found {len(results)} results\n")

        # Fetch all pages in parallel
        pages_results = [None] * len(results)
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = {
                executor.submit(fetch_and_rank, (idx, r)): idx
                for idx, r in enumerate(results)
            }
            for future in as_completed(futures):
                res = future.result()
                pages_results[res["idx"]] = res

        final_output = []
        for res in pages_results:
            if res is None:
                continue
            print(f"{res['idx']+1}) {res['title']}\n- {res['href']}\n- {res['body']}\n")

            entry = f"[{res['title']}]\n{res['body']}"
            if res['context']:
                entry += f"\n{res['context']}"
            final_output.append(entry)

        return "\n\n".join(final_output)

    except Exception as ex:
        return f"Search error: {ex}"

print("Knowledge tools loaded")

## 10 · System prompt del SingleAgent

Un unico system prompt che descrive tutti i tool disponibili e guida l'agente
a scegliere autonomamente quello corretto in base al tipo di domanda.
Non esiste più un orchestratore separato: l'agente decide da solo.

In [ ]:
# ── Tutti i tool in un'unica lista ───────────────────────────────────────
all_tools = [
    calculator,
    modular_arithmetic,
    statistics_helper,
    python_code_runner,
    wikipedia_search,
    duckduckgo_search,
]

ALL_TOOLS_DESC = render_text_description(all_tools)

SINGLE_AGENT_SYSTEM_PROMPT = f"""
You are a Quiz Game Specialist Agent with access to both mathematical and factual tools.
Your ONLY job is to answer multiple-choice quiz questions accurately by using the right tool.

AVAILABLE TOOLS:
{ALL_TOOLS_DESC}

TOOL SELECTION RULES — choose ONE tool per question:

  MATH questions (arithmetic, algebra, calculus, statistics, probability,
  modular arithmetic, combinatorics, geometry — anything requiring computation):
    → Try python_code_runner first; fall back to calculator or other math tools if it fails.
    → NEVER compute in your head — always delegate to a math tool.

  KNOWLEDGE questions (history, science, geography, culture, current events,
  famous people, definitions — anything NOT purely computational):
    → Use wikipedia_search for stable facts.
    → Use duckduckgo_search for recent or current events.
    → If Wikipedia returns no useful result, fall back to duckduckgo_search.
    → NEVER answer from internal memory — always verify with a search tool first.

WORKFLOW:
1. Read the question and classify it (math vs knowledge).
2. Call the appropriate tool with a precise query or expression.
3. Analyse the tool result and match it against the answer choices.
4. If necessary (e.g. tool error or ambiguous result), call a second tool.
5. Output EXACTLY one character: A, B, C, or D. No other text, no explanation.

IMPORTANT: Use the EXACT function name and parameter names as shown above.
Do NOT write tool calls as plain text or JSON. Use the native tool-calling mechanism.
"""

print("System prompt created")

## 11 · SingleAgent

Ri-esegui questa cella per cambiare modello senza riavviare il kernel.

Un unico agente ReAct con accesso a tutti i tool (matematici + ricerca).
Non esistono più sub-agenti né orchestratore.

In [ ]:
# LangGraph's create_react_agent handles the ReAct loop internally.

single_agent = create_agent(
    model=make_model(temperature=0.0, model_name=MODEL_NAME),
    system_prompt=SINGLE_AGENT_SYSTEM_PROMPT,
    tools=all_tools,
    #debug=True,
)

print("SingleAgent ready")

## 12 · Wrapper quiz

Stesso wrapper del multi-agent, semplificato: non c'è più `question_trace` per-agente.
La traccia ora ha un'unica chiave `"agent"` invece di `"orchestrator"` / `"math"` / `"knowledge"`.

In [ ]:
def format_question(question: str, choices: dict, competition) -> str:
    """Formatta domanda + opzioni come stringa per il SingleAgent."""
    competition = f"COMPETITION {competition}:\n"
    lines = [competition + question, "Options:"]
    for letter, text in choices.items():
        lines.append(f"{letter}: {text}")
    lines.append("\n")
    return "\n".join(lines)


def extract_letter(raw: str) -> str:
    """
    Estrae A/B/C/D dalla risposta grezza.
    Fallback: 'A'.
    """
    m = re.search(r"\b([A-Da-d])\b", raw.strip())
    return m.group(1).upper() if m else "A"


def answer_quiz_question(question: str, choices: dict, competition) -> dict:
    """
    Punto di ingresso per il quiz. Invoca il SingleAgent e raccoglie la traccia.

    Parameters
    ----------
    question    : str  — testo della domanda
    choices     : dict — {'A': '...', 'B': '...', 'C': '...', 'D': '...'}
    competition : str  — nome/id della competizione (per il prompt)

    Returns
    -------
    dict con campi:
        answer_letter : str  — lettera scelta (A/B/C/D)
        answer_text   : str  — testo dell'opzione scelta
        raw           : str  — risposta grezza del SingleAgent
        agent_trace   : dict — traccia con struttura:
            {
              "agent": {"model": ..., "raw_output": ..., "tools_called": [...]}
            }
    """
    prompt = format_question(question, choices, competition)
    result = single_agent.invoke(
        {"messages": [("human", prompt)]},
        config={"recursion_limit": 30},
    )
    trace = extract_agent_trace(result)
    print_agent_stats(result, agent_name="SingleAgent")

    raw    = result["messages"][-1].content
    letter = extract_letter(raw)
    return {
        "answer_letter": letter,
        "answer_text":   choices.get(letter, "?"),
        "raw":           raw,
        "agent_trace":   {"agent": trace},
    }

## 13 · Test

Testa il sistema su domande note prima di avviare il game loop.

In [ ]:
# ── Test di conoscenza (DuckDuckGo) ──────────────────────────────────────────
q_knowledge = "Who won the general classification of the Tour de France in the year 2025?"
c_knowledge = {
    "A": "Jonas Vingegaard",
    "B": "Tadej Pogačar",
    "C": "Remco Evenepoel",
    "D": "Primož Roglič",
}
res = answer_quiz_question(q_knowledge, c_knowledge, "sport")
print(f"\nANSWER: {res['answer_letter']} → {c_knowledge.get(res['answer_letter'], '?')}")
print(f"Agent trace keys: {list(res['agent_trace'].keys())}")

In [ ]:
# ── Test matematico (calculator tool) ────────────────────────────────────────
q_math = "What is the value of sqrt(144) + 3 squared?"
c_math = {"A": "18", "B": "21", "C": "15", "D": "27"}
res_math = answer_quiz_question(q_math, c_math, "math")
print(f"\nANSWER: {res_math['answer_letter']} → {c_math.get(res_math['answer_letter'], '?')}")
print(f"Agent trace keys: {list(res_math['agent_trace'].keys())}")

## 14 · Game loop

Esegue `NUM_RUNS` partite per ogni competizione e salva i risultati in `data/runs/`.

**Struttura del log per domanda** (compatibile con `analysis.ipynb`):
```json
{
  "level": 3,
  "prize": 500,
  "question_text": "...",
  "options": {"A": "...", "B": "...", "C": "...", "D": "..."},
  "llm_raw_output": "B",
  "agent_trace": {
    "agent": {"model": "phi4-mini", "raw_output": "B", "tools_called": [...]}
  },
  "chosen_letter": "B",
  "correct": true,
  "timed_out": false,
  "time": 1.83
}
```

In [ ]:
# ── Crea la cartella del run e salva i metadati ───────────────────────────────
run_dir  = create_run_folder(MODEL_NAME)
metadata = save_metadata(run_dir, MODEL_NAME, REASONING, RUN_COMMENT, TECHNIQUES, NUM_RUNS)

comps = client.competitions.list_all()
print("Competitions found:", len(comps))
all_comp_data = {}

LETTERS = ["A", "B", "C", "D"]

for comp in comps:

    #if comp.id != 3:
    #    continue

    print()
    print("=" * 60)
    print("Competition", comp.id, ":", comp.name)
    attempts_log = []

    for run_idx in range(1, NUM_RUNS + 1):
        print("  Run", run_idx, "/", NUM_RUNS, "...", end=" ", flush=True)
        session       = client.game.start(comp.id)
        questions_log = []
        max_time      = 0.0

        last_correct_level = 0
        final_earned       = 0

        while session.in_progress:
            question = session.current_question
            if question is None:
                break

            opts         = question.options[:4]
            options_dict = {LETTERS[i]: opts[i].text for i in range(len(opts))}

            # Livello e premio dalla GameState (question.level può essere 0)
            current_level = session.state.current_level
            current_prize = next(
                (ml.amount for ml in session.state.money_pyramid if ml.level == current_level),
                None,
            )

            t0     = time.time()
            result = answer_quiz_question(question.text, options_dict, comp.name)
            chosen_idx = LETTERS.index(result["answer_letter"])
            chosen_opt = question.options[chosen_idx]
            answer     = session.answer(chosen_opt.id)
            elapsed    = time.time() - t0
            if elapsed > max_time:
                max_time = elapsed

            if answer.correct:
                last_correct_level = current_level
                final_earned       = current_prize

            questions_log.append({
                "level":          current_level,
                "prize":          current_prize,
                "question_text":  question.text,
                "options":        options_dict,
                "llm_raw_output": result["raw"],         # compatibilità celle analisi
                "agent_trace":    result["agent_trace"],  # traccia agente
                "chosen_letter":  result["answer_letter"],
                "correct":        answer.correct,
                "timed_out":      answer.timed_out,
                "time":           elapsed,
            })

        print(
            "earned=" + str(final_earned)
            + ", q=" + str(len(questions_log))
            + ", max_t=" + f"{max_time:.2f}s"
        )
        attempts_log.append({
            "attempt":     run_idx,
            "earned":      final_earned,
            "final_level": last_correct_level,
            "questions":   questions_log,
        })

    save_competition_result(run_dir, comp, attempts_log)
    all_comp_data[comp.id] = {
        "name":       comp.name,
        "max_levels": comp.max_levels,
        "attempts":   attempts_log,
    }

run_info = save_run_summary(run_dir, all_comp_data, metadata)
update_runs_log(RUNS_DIR, run_dir.name, run_info)
print()
print("Run complete! Overall accuracy:", f"{run_info['overall_accuracy']:.1%}")